# dbt: tests, unit tests and contracts

dbt gives you three different safety nets. This notebook triggers each one **on purpose** so you see what a failure looks like:

| Layer | Checks | When it runs |
|---|---|---|
| **Data tests** (`unique`, `not_null`, singular SQL, ...) | the *data* in a built table | after the model is built (`dbt test`, `dbt build`) |
| **Unit tests** | the *logic* of a model, with mocked inputs | before/without real data (`dbt test --select test_type:unit`) |
| **Contracts** | the *shape* (column names and types) of a model | while building the model |

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
from lab_utils import dbt, trino, reset_dbt_demo, DBT_PROJECT

reset_dbt_demo()
dbt("build");

## 1. Data tests
Generic tests are declared next to the model in `schema.yml`; a **singular test** is a SQL file in `tests/` that returns the offending rows.

In [ ]:
print((DBT_PROJECT / "models/marts/schema.yml").read_text().split("unit_tests:")[0])
print("-" * 70)
print((DBT_PROJECT / "tests/assert_no_negative_amounts.sql").read_text())

In [ ]:
dbt("test", "--select", "fct_orders")

### Make a test fail, and keep the failing rows with `--store-failures`
A bad order (negative quantity) reaches the source. The incremental run merges it into `fct_orders`, and the singular test catches it.
With `--store-failures`, dbt saves the rows that failed each test as an (Iceberg) table so you can inspect them with SQL.

In [ ]:
trino("INSERT INTO iceberg.dbt_demo_raw.raw_orders VALUES (90, 2, 3, -2, TIMESTAMP '2026-09-22 08:00:00', 'placed')")
dbt("run", "--select", "fct_orders")
result = dbt("test", "--select", "fct_orders", "--store-failures")

In [ ]:
audit = trino("SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema LIKE '%dbt_test__audit%' ORDER BY table_name")
display(audit)
failing = trino("SELECT * FROM iceberg.dbt_test__audit.assert_no_negative_amounts")     # rows that violated the singular test
print(f"{len(failing)} failing row(s) stored for assert_no_negative_amounts:")
failing

Fix the source, then rebuild. **Note:** an incremental `merge` model never deletes rows that disappear upstream, so the bad row stays in `fct_orders` until a full refresh.

In [ ]:
trino("DELETE FROM iceberg.dbt_demo_raw.raw_orders WHERE order_id = 90")
dbt("run", "--select", "fct_orders", "--full-refresh")
dbt("test", "--select", "fct_orders");

## 2. Unit tests
The unit test in `models/marts/schema.yml` feeds mocked rows into `customer_summary` (customers with orders just below, exactly at and above the tier thresholds) and compares the output with the expected rows.
It needs no real data and guards the business rule in the macro `spend_tier`.

In [ ]:
print("unit_tests:" + (DBT_PROJECT / "models/marts/schema.yml").read_text().split("unit_tests:")[1])
print("-" * 70)
print((DBT_PROJECT / "macros/spend_tier.sql").read_text())

In [ ]:
dbt("test", "--select", "test_type:unit")

Now **break the business rule** (silver starts at 600 instead of 500) and run the unit test again. The macro is restored afterwards.
dbt prints a diff of the expected and the actual rows.

In [ ]:
macro = DBT_PROJECT / "macros/spend_tier.sql"
original = macro.read_text()
try:
    macro.write_text(original.replace(">= 500", ">= 600"))
    dbt("test", "--select", "test_type:unit", show_log=True)
finally:
    macro.write_text(original)
print("macro restored:", macro.read_text() == original)

## 3. Contracts
`dim_products` declares `contract: {enforced: true}` with a `data_type` for each column. dbt compares the model's real output with the contract **before** creating the table:

In [ ]:
dbt("run", "--select", "dim_products")

A model that drifts from its contract (wrong type and a missing column) fails with a clear table of mismatches instead of silently changing the shape downstream users depend on:

In [ ]:
bad_sql = DBT_PROJECT / "models/marts/dim_products_bad.sql"
bad_yml = DBT_PROJECT / "models/marts/dim_products_bad.yml"
bad_sql.write_text("{{ config(materialized='table', contract={'enforced': True}) }}\n"
                   "select cast(product_id as bigint) as product_id, product_name from {{ ref('stg_products') }}\n")
bad_yml.write_text("version: 2\nmodels:\n  - name: dim_products_bad\n    columns:\n"
                   "      - {name: product_id, data_type: integer}\n      - {name: product_name, data_type: varchar}\n"
                   "      - {name: category, data_type: varchar}\n")
try:
    result = dbt("run", "--select", "dim_products_bad")
finally:
    bad_sql.unlink(); bad_yml.unlink()

## 4. Selecting what to run
`dbt ls` lists what a selection matches without running anything. Selectors: `+model` (with its parents), `model+` (with its children), `test_type:unit`, `resource_type:...`.

In [ ]:
print("--- customer_summary and everything it depends on")
_ = dbt("ls", "--select", "+customer_summary", "--resource-type", "model", "--resource-type", "seed")
print("--- the models that depend on stg_orders")
_ = dbt("ls", "--select", "stg_orders+", "--resource-type", "model")

## Notes
* `dbt build` runs tests as part of the DAG, so a failing test **stops downstream models** (see notebook 02).
* Tests can be downgraded with `severity: warn` in their config, and data tests can be scoped with `where`.
* Unit tests need dbt 1.8 or newer and work with dbt-trino on this lab's Trino 483.